In [1]:
# Cell 1a
import sys
sys.stderr = open("mcp_server.log", "w")

In [2]:
# Cell 1b
##import sys

##MCP_DIAGNOSTIC_LOG = open("mcp_server.log", "w", encoding="utf-8")
##sys.stderr = MCP_DIAGNOSTIC_LOG
##print("MCP server diagnostics will be written to mcp_server.log")

In [3]:
# Cell 1c
!pip install -q  fastmcp

In [4]:
# Cell 2
%%writefile trip_server.py
# Server preview only: not executed in this activity.
from fastmcp import FastMCP

mcp = FastMCP("TripUtilities")

@mcp.tool()
def estimate_daily_budget(days: int, daily_cost: float) -> float:
    """Estimate a trip budget from duration and daily cost."""
    if days < 0 or daily_cost < 0:
        raise ValueError("Inputs must be nonnegative")
    return days * daily_cost


@mcp.resource("trip://notes/helsinki")
def helsinki_notes() -> str:
    """Return illustrative planning notes for a city visit."""
    return (
        "Illustrative trip notes (not live travel data).\n"
        "City: Helsinki\n"
        "Interests: museums, architecture, and cafés.\n"
        "Preference: walk between nearby places where practical.\n"
        "Reminder: verify opening hours, accessibility, and routes."
    )

@mcp.prompt()
def outline_day_trip(city: str, focus: str = "museums") -> str:
    """Provide a reusable request for a one-day itinerary."""
    return (
        f"Prepare a one-day itinerary for {city}, focusing on {focus}. "
        "Use verified sources for opening hours and travel times; "
        "label unverified details as uncertain."
    )

if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing trip_server.py


In [ ]:
# Cell 3a
import sys
from pathlib import Path
from fastmcp import Client
from fastmcp.client.transports import StdioTransport

transport = StdioTransport(
    command=sys.executable,
    args=[str(Path("trip_server.py").resolve())]
)

client = Client(transport)

async with client:
    # Discover tools
    tools = await client.list_tools()
    print("Available tools:", [t.name for t in tools])

    # Call tool through MCP
    result = await client.call_tool(
        "estimate_daily_budget",
        {"days": 5, "daily_cost": 120.0}
    )

    print("Budget:", result.data)

    if not result.is_error:
      print(result.structured_content)

Available tools: ['estimate_daily_budget']
Budget: 600.0
{'result': 600.0}


In [ ]:

# Cell 3b FastMCP Client
# Discover and use tools, resources, and prompts

import sys
from pathlib import Path
from fastmcp import Client
from fastmcp.client.transports import StdioTransport

transport = StdioTransport(
    command=sys.executable,
    args=[str(Path("trip_server.py").resolve())]
)

client = Client(transport)

async with client:

    # 1. TOOLS: discover and execute
    print("=== AVAILABLE TOOLS ===")
    tools = await client.list_tools()

    for tool in tools:
        print(f"{tool.name}: {tool.description}")

    result = await client.call_tool(
        "estimate_daily_budget",
        {"days": 5, "daily_cost": 120.0}
    )
    print("Budget:", result.data)

    # 2. RESOURCES: discover and read
    print("\n=== AVAILABLE RESOURCES ===")
    resources = await client.list_resources()

    for resource in resources:
        print(f"{resource.name}: {resource.uri}")

    resource_content = await client.read_resource(
        "trip://notes/helsinki"
    )

    notes = "\n".join(
        item.text
        for item in resource_content
        if hasattr(item, "text")
    )
    print("\nResource content:\n", notes)

    # 3. PROMPTS: discover and render
    print("\n=== AVAILABLE PROMPTS ===")
    prompts = await client.list_prompts()

    for prompt in prompts:
        print(f"{prompt.name}: {prompt.description}")

    prompt_result = await client.get_prompt(
        "outline_day_trip",
        {"city": "Helsinki", "focus": "museums"}
    )

    prompt_text = "\n".join(
        msg.content.text
        for msg in prompt_result.messages
        if hasattr(msg.content, "text")
    )
    print("\nRendered prompt:\n", prompt_text)

    # 4. USE RESOURCE + PROMPT TOGETHER
    llm_input = f"""
Context:
{notes}

Task:
{prompt_text}
"""
    print("\n=== COMBINED LLM INPUT ===")
    print(llm_input)


=== AVAILABLE TOOLS ===
estimate_daily_budget: Estimate a trip budget from duration and daily cost.
Budget: 600.0

=== AVAILABLE RESOURCES ===
helsinki_notes: trip://notes/helsinki

Resource content:
 Illustrative trip notes (not live travel data).
City: Helsinki
Interests: museums, architecture, and cafés.
Preference: walk between nearby places where practical.
Reminder: verify opening hours, accessibility, and routes.

=== AVAILABLE PROMPTS ===
outline_day_trip: Provide a reusable request for a one-day itinerary.

Rendered prompt:
 Prepare a one-day itinerary for Helsinki, focusing on museums. Use verified sources for opening hours and travel times; label unverified details as uncertain.

=== COMBINED LLM INPUT ===

Context:
Illustrative trip notes (not live travel data).
City: Helsinki
Interests: museums, architecture, and cafés.
Preference: walk between nearby places where practical.
Reminder: verify opening hours, accessibility, and routes.

Task:
Prepare a one-day itinerary for H